# 02. YOLO11: обучение и тест детектора

Детектор обучается только на полных изображениях Gunduz. Долгое обучение и однократный test включаются раздельно.

In [ ]:
from pathlib import Path
import yaml
from core.notebook_runtime import bootstrap_notebook, describe_runtime, gpu_preflight, run_guarded

GPU_INDEX = 0
YOLO_BATCH = 4
YOLO_IMAGE_SIZE = 1024
WORKERS = 4
context = bootstrap_notebook(gpu_index=GPU_INDEX)
PROJECT_ROOT = context.project_root
CONFIG = PROJECT_ROOT / 'configs/detector.yaml'
WEIGHTS = PROJECT_ROOT / 'artifacts/detector/yolo11m/weights/best.pt'
RUN_TRAINING = False
RUN_FINAL_TEST = False
ENABLE_CLEARML = False
describe_runtime(context)
gpu_preflight(context, minimum_vram_gb=8.0)

## Effective configuration и данные

In [ ]:
from core.config_loader import load_config
overrides = [f'training.batch={YOLO_BATCH}', f'training.imgsz={YOLO_IMAGE_SIZE}', f'training.workers={WORKERS}']
config = load_config(CONFIG, overrides=overrides)
data_yaml = PROJECT_ROOT / config['paths']['data_yaml']
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
assert data_yaml.is_file(), f'Сначала выполните 01_prepare_data.ipynb: {data_yaml}'
data_config = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))
for split in ('train', 'val', 'test'):
    split_file = Path(data_config['path']) / data_config[split]
    assert split_file.is_file() and split_file.read_text(encoding='utf-8').strip(), split_file
    print(split, 'images:', len(split_file.read_text(encoding='utf-8').splitlines()))
print('YOLO split preflight: OK')

## Обучение

In [ ]:
train_command = context.module_command('scripts.run_train_detector', '--config', str(CONFIG))
for value in overrides:
    train_command += ['--set', value]
if ENABLE_CLEARML:
    train_command += ['--set', 'clearml.enabled=true']
if RUN_TRAINING:
    assert not WEIGHTS.exists(), f'Checkpoint уже существует: {WEIGHTS}'
run_guarded(context, train_command, enabled=RUN_TRAINING, label='yolo-train')

## Однократный test
Не используйте test для подбора гиперпараметров.

In [ ]:
test_command = context.module_command('scripts.run_test_detector', '--config', str(CONFIG), '--weights', str(WEIGHTS))
for value in overrides:
    test_command += ['--set', value]
if ENABLE_CLEARML:
    test_command += ['--set', 'clearml.enabled=true']
if RUN_FINAL_TEST:
    assert WEIGHTS.is_file(), WEIGHTS
run_guarded(context, test_command, enabled=RUN_FINAL_TEST, label='yolo-final-test')

## Артефакты

In [ ]:
output = PROJECT_ROOT / config['paths']['output_dir']
for path in sorted(output.rglob('*')) if output.exists() else []:
    if path.is_file(): print(path.relative_to(PROJECT_ROOT))